# センサーの最適配置（元の課題）を Fixstars Amplify AE で解き、古典的手法と比べる

- 古典: MILP（HiGHS、厳密解）と貪欲法
- QUBO 1: マスの被覆変数＋Amplify の制約（課題をそのまま表す。スラック変数あり）
- QUBO 2: ラグランジュ緩和（スラック変数なし）＋乗数の外側ループ

式は `placement/README.md` にまとめてあります。

In [ ]:
!pip install -q amplify dimod dwave-neal
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!git clone -q -b claude/request-5v4fsc https://github.com/mori560269/Q-Questgrow-.git
%cd Q-Questgrow-/placement
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
except Exception:
    pass

In [ ]:
from getpass import getpass
token = getpass('Amplify のアクセストークン ')

## 古典的手法

In [ ]:
import sys, numpy as np
sys.path.insert(0, '.')
from common import Problem, save_map
from classical import solve_milp, solve_greedy, repair
prob = Problem(cell_m=1000)
x_milp, info = solve_milp(prob)
ev_milp = prob.evaluate(x_milp, cell_m=1000)
print('MILP', ev_milp['cost'], ev_milp['feasible'], round(info['sec'], 1), '秒')
x_gr, info = solve_greedy(prob)
ev_gr = prob.evaluate(x_gr, cell_m=1000)
print('貪欲法', ev_gr['cost'], ev_gr['feasible'], round(info['sec'], 1), '秒')

## QUBO 1: Amplify の制約で課題をそのまま表す（AE で解く）

In [ ]:
from qubo_amplify import build_model, solve_model
pq = Problem(cell_m=2000)
model, xv, binfo = build_model(pq)
print(binfo)
sols, sinfo = solve_model(model, xv, backend='amplify', token=token, time_limit_ms=10000)
raw = [prob.evaluate(s, cell_m=1000) for s in sols]
best_raw = min(raw, key=lambda r: (not r['feasible'], r['cost']))
print('AE の生の解', best_raw['cost'], best_raw['feasible'], best_raw['checks'], sinfo)
x_q1 = repair(prob, np.array(best_raw['x'], float))
ev_q1 = prob.evaluate(x_q1, cell_m=1000)
print('QUBO 1 + 修復', ev_q1['cost'], ev_q1['feasible'])

## QUBO 2: ラグランジュ緩和（内側の QUBO を AE で 50 回解く。1 回 1 秒）

In [ ]:
from qubo_amplify import solve_lagrangian
best, linfo = solve_lagrangian(prob, iters=50, backend='amplify', token=token, time_limit_ms=1000)
ev_q2 = prob.evaluate(best[1], cell_m=1000) if best else None
print('QUBO 2', None if best is None else best[0], '実行可能な反復', linfo['feasible_iters'], '/ 50')
x_q2p = repair(prob, best[1]) if best else None
ev_q2p = prob.evaluate(x_q2p, cell_m=1000) if best else None
print('QUBO 2 + 修復', None if best is None else ev_q2p['cost'])

## まとめ

In [ ]:
import pandas as pd
rows = [('MILP（最適）', ev_milp), ('貪欲法', ev_gr), ('QUBO 1 生の解（AE）', best_raw),
        ('QUBO 1 + 修復（AE）', ev_q1)]
if ev_q2:
    rows += [('QUBO 2（AE）', ev_q2), ('QUBO 2 + 修復（AE）', ev_q2p)]
opt = ev_milp['cost']
pd.DataFrame([dict(手法=n, コスト=e['cost'], 最適との差=f"{(e['cost']-opt)/opt:+.1%}", 実行可能=e['feasible'],
                   **e['n_sensors']) for n, e in rows])

In [ ]:
from IPython.display import Image
save_map(prob, {n: e for n, e in rows if e['feasible']}, 'placement_maps_ae.png')
Image('placement_maps_ae.png')